# Training Benchmark and Tuning

Trainer mode: we will tune with evidence, not guesswork.

**Outcome:** compare a baseline training step and a tuned step.

## Step 1 - Tiny setup

Ask learners: what metric matters more here, latency or throughput?

In [ ]:
import time
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.nn.Linear(512, 512).to(device)

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

## Step 2 - Baseline batch

Expected: this gives our first reference step time.

In [ ]:
x = torch.randn((32, 512), device=device)
y = torch.randn((32, 512), device=device)

In [ ]:
t0 = time.perf_counter()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()

In [ ]:
opt.step()
opt.zero_grad()
base_time = time.perf_counter() - t0
print(round(base_time, 4))

## Step 3 - Tuned batch size

Common mistake: tuning without resetting the comparison setup.

In [ ]:
x2 = torch.randn((64, 512), device=device)
y2 = torch.randn((64, 512), device=device)

In [ ]:
t1 = time.perf_counter()
loss_fn(model(x2), y2).backward()
opt.step()
opt.zero_grad()

In [ ]:
tuned_time = time.perf_counter() - t1
print(round(tuned_time, 4))

## Checkpoint — Did larger batch help? What risk did we introduce?

**The questions we asked:**
- Did larger batch help throughput here?
- What memory risk did we introduce?

**Model answer (strong understanding)**

Larger batch sizes usually improve *throughput* (samples per second) on GPU because you amortize kernel launch overhead, better saturate the tensor cores, and get more efficient memory access patterns. In this lab you should have seen a clear increase in the "samples per second" metric when you moved from the tiny baseline batch to something more realistic (often 32–128 depending on the tiny model and GPU).

The memory risk is real and immediate: every sample in the batch needs its own forward and backward activations stored for the optimizer step. At some point you will hit an Out-Of-Memory (OOM) error. The exact crossover point depends on model size, sequence length (for transformers), precision, and whether you are using activation checkpointing or gradient accumulation.

The sophisticated move is not "always use the biggest batch that fits." It is "use the largest batch that still gives me good generalization, then use gradient accumulation or other tricks if I need an even larger *effective* batch for optimization reasons."

**If you saw almost no throughput gain from bigger batch, the usual causes are:**
- The model was so tiny that even the small batch already saturated the GPU.
- You were still on CPU (easy to miss when experimenting).
- The data loading or the logging was the actual bottleneck, not the forward/backward pass.

**Common misconception that bites people in production**

"Bigger batch = faster training = better. I will just set batch size to 1024 everywhere."

Two problems: (1) at some point you OOM or the generalization gets worse (the noise in the gradient is actually useful for escaping sharp minima), and (2) on real large models the relationship between batch size and wall-clock time is not monotonic once you include data loading, all-reduce communication in multi-GPU, and memory fragmentation.

**If you are feeling lost about what number to actually ship with**

That feeling is correct and healthy. The lab gave you the *measurement skill*. Choosing the production batch size is a later, separate decision that also involves validation metrics, not just speed. You now have the tool to run the experiments that inform that decision instead of guessing.


## Lesson Recap — What You Actually Learned

- Throughput (samples/sec) and latency per step are different numbers and you must decide which one your users or your training budget actually cares about.
- Larger batches are usually more efficient on GPU up to the point where you either OOM or generalization suffers.
- Mixed precision (AMP) is one of the highest-leverage, lowest-risk wins available in 2026 for both speed and memory.
- You now have a repeatable micro-benchmark ritual you can use to evaluate any "I heard this trick makes training faster" claim with actual data instead of folklore.

**Human note:** If you are currently training models and you have never done this kind of controlled before/after measurement, you have been shipping on hope. The fact that you just did it for a tiny model means you can now do it for the real ones. That is a genuine professional upgrade.


## Role Lens — Why This Matters in Real Work

**DevOps / MLOps** — When a data scientist says "we need 4× more GPUs for the new model," your first question after looking at utilization should be: "Have we measured what happens if we double the batch size or turn on AMP?" The 30-minute experiment you just learned is how you push back on expensive hardware requests with data instead of politics.

**Data Science** — Hyperparameter sweeps that ignore hardware efficiency are how you burn $18k on a weekend. Every time you try a new optimizer, learning rate schedule, or architecture, you should be able to answer "did this change also change the hardware efficiency?" The micro-benchmark + interpretation habit is now part of your scientific method.

**Data Engineering** — Many modern feature stores and training data pipelines have GPU stages. The same batch-size and precision questions appear when you are deciding whether to materialize transformed batches on the GPU or stream them. The measurement discipline transfers directly.

---

**You can now tune with evidence instead of folklore.**

Next we take these skills into the data engineering world — ETL bottlenecks, cuDF vs pandas, GPU SQL, and pipelines that actually finish on the data you have today instead of the data you wish you had. You are ready.
